# 02 USDA Processing

Process the MVP USDA inputs only.

Inputs: `food.csv`, `food_nutrient.csv`, `nutrient.csv`, `food_category.csv`, `foundation_food.csv`

Output: `data/processed/usda_food_clean.parquet`

In [ ]:
from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
USDA_DIR = ROOT / 'USDA'
OUTPUT_DIR = ROOT / 'data' / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'usda_food_clean.parquet'

FINAL_COLUMNS = [
    'food_id', 'food_name_en', 'food_name_ar', 'food_group', 'meal_types',
    'serving_name', 'serving_weight_g', 'nutrition_basis', 'calories',
    'protein', 'carbs', 'fat', 'fiber', 'diet_tags', 'allergens',
    'is_composite_dish', 'source'
]

def stable_id(prefix: str, value: object) -> str:
    digest = hashlib.sha1(str(value).encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{digest}'

def normalize_food_name(value: object) -> str:
    if pd.isna(value):
        return ''
    text = str(value).strip().lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s*,\s*', ', ', text)
    return text.strip(' ,')

def map_usda_category(category: object, name: str) -> str:
    category_text = '' if pd.isna(category) else str(category).lower()
    name = name.lower()
    if any(k in name for k in ['beef', 'pork', 'lamb', 'chicken', 'turkey', 'fish', 'salmon', 'tuna', 'shrimp', 'sausage', 'ham', 'bacon']):
        return 'Protein'
    if 'beverage' in category_text or any(k in name for k in ['juice', 'drink', 'tea', 'coffee', 'soda']):
        return 'Beverages'
    if 'fruit' in category_text:
        return 'Beverages' if 'juice' in name else 'Fruits'
    if 'vegetable' in category_text:
        return 'Vegetables'
    if any(k in category_text for k in ['cereal', 'grain', 'pasta', 'baked']):
        return 'Grains'
    if 'legume' in category_text or any(k in name for k in ['bean', 'lentil', 'chickpea', 'pea']):
        return 'Legumes'
    if 'dairy' in category_text or 'egg' in category_text:
        return 'Dairy'
    if any(k in category_text for k in ['beef', 'poultry', 'pork', 'lamb', 'finfish', 'shellfish', 'sausages']):
        return 'Protein'
    if 'fat' in category_text or 'oil' in category_text or 'nut' in category_text or 'seed' in category_text:
        return 'Healthy Fats'
    if any(k in category_text for k in ['sweets', 'snacks']):
        return 'Snacks'
    if any(k in category_text for k in ['meals', 'fast foods', 'restaurant', 'soups', 'sauces']):
        return 'Composite Dish'
    return 'Composite Dish'

def is_composite(name: str, category: str) -> bool:
    markers = [' with ', 'prepared', 'meal', 'entree', 'soup', 'stew', 'sandwich', 'pizza', 'fried rice']
    return category == 'Composite Dish' or any(marker in name for marker in markers)

def infer_meal_types(group: str) -> str:
    if group in {'Grains', 'Dairy', 'Legumes'}:
        return 'Breakfast|Lunch|Dinner'
    if group in {'Protein', 'Composite Dish'}:
        return 'Lunch|Dinner'
    if group in {'Fruits', 'Snacks', 'Beverages'}:
        return 'Snack'
    return 'Lunch|Dinner'

def infer_allergens(name: str) -> str:
    allergens = []
    if any(k in name for k in ['milk', 'cheese', 'yogurt', 'cream', 'butter']):
        allergens.append('milk')
    if 'egg' in name:
        allergens.append('egg')
    if any(k in name for k in ['fish', 'salmon', 'tuna', 'shrimp', 'shellfish']):
        allergens.append('fish')
    if any(k in name for k in ['wheat', 'bread', 'pasta', 'macaroni', 'flour']):
        allergens.append('wheat')
    if any(k in name for k in ['peanut', 'almond', 'walnut', 'cashew']):
        allergens.append('tree_nuts_or_peanuts')
    return '|'.join(allergens)

def infer_diet_tags(group: str, name: str) -> str:
    if group in {'Grains', 'Vegetables', 'Fruits', 'Legumes', 'Healthy Fats', 'Beverages'} and not any(k in name for k in ['meat', 'beef', 'chicken', 'fish', 'egg', 'milk', 'cheese', 'yogurt']):
        return 'plant_based'
    if group in {'Protein', 'Dairy'}:
        return 'animal_based'
    if group == 'Composite Dish':
        return 'composite'
    return ''

food = pd.read_csv(USDA_DIR / 'food.csv', low_memory=False)
food_nutrient = pd.read_csv(USDA_DIR / 'food_nutrient.csv', low_memory=False)
nutrient = pd.read_csv(USDA_DIR / 'nutrient.csv', low_memory=False)
food_category = pd.read_csv(USDA_DIR / 'food_category.csv', low_memory=False)
foundation_food = pd.read_csv(USDA_DIR / 'foundation_food.csv', low_memory=False)

food = food.merge(food_category[['id', 'description']], left_on='food_category_id', right_on='id', how='left', suffixes=('', '_category'))
food['is_foundation'] = food['fdc_id'].isin(foundation_food['fdc_id'])

selected_food = food[food['data_type'].eq('foundation_food') | food['is_foundation']].copy()
if selected_food.empty:
    selected_food = food[food['data_type'].isin(['foundation_food', 'sample_food'])].copy()

nutrient_targets = {
    1008: 'calories',
    2047: 'calories',
    2048: 'calories',
    1003: 'protein',
    1005: 'carbs',
    1004: 'fat',
    1079: 'fiber',
}
nutrient_lookup = nutrient[['id', 'name', 'unit_name']].copy()
facts = food_nutrient[food_nutrient['fdc_id'].isin(selected_food['fdc_id'])].copy()
facts = facts[facts['nutrient_id'].isin(nutrient_targets)].copy()
facts['target_nutrient'] = facts['nutrient_id'].map(nutrient_targets)
facts = facts.merge(nutrient_lookup, left_on='nutrient_id', right_on='id', how='left')
facts['amount'] = pd.to_numeric(facts['amount'], errors='coerce')
facts = facts.sort_values(['fdc_id', 'target_nutrient', 'nutrient_id'])
facts = facts.drop_duplicates(['fdc_id', 'target_nutrient'], keep='first')
pivot = facts.pivot(index='fdc_id', columns='target_nutrient', values='amount').reset_index()

clean = selected_food.merge(pivot, on='fdc_id', how='inner')
for col in ['calories', 'protein', 'carbs', 'fat', 'fiber']:
    if col not in clean.columns:
        clean[col] = np.nan
    clean[col] = pd.to_numeric(clean[col], errors='coerce').fillna(0).clip(lower=0)

clean['food_name_en'] = clean['description'].map(normalize_food_name)
clean = clean[clean['food_name_en'].ne('')].copy()
clean['food_group'] = [map_usda_category(cat, name) for cat, name in zip(clean['description_category'], clean['food_name_en'])]
clean['name_key'] = clean['food_name_en'].str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip()
clean = clean.sort_values(['is_foundation', 'publication_date'], ascending=[False, False]).drop_duplicates('name_key', keep='first')

clean['food_id'] = clean['fdc_id'].map(lambda value: stable_id('usda', value))
clean['food_name_ar'] = ''
clean['meal_types'] = clean['food_group'].map(infer_meal_types)
clean['serving_name'] = '100 g'
clean['serving_weight_g'] = 100.0
clean['nutrition_basis'] = 'per_100g'
clean['diet_tags'] = [infer_diet_tags(group, name) for group, name in zip(clean['food_group'], clean['food_name_en'])]
clean['allergens'] = clean['food_name_en'].map(infer_allergens)
clean['is_composite_dish'] = [is_composite(name, group) for name, group in zip(clean['food_name_en'], clean['food_group'])]
clean['source'] = 'usda_foundation'

clean = clean[FINAL_COLUMNS].copy()
clean.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')

print(f'Selected USDA rows: {len(selected_food)}')
print(f'Clean USDA rows exported: {len(clean)}')
print(f'Output: {OUTPUT_PATH}')
print(clean.head(10))
